In [3]:
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv("/content/Liar_Dataset.csv")
df.head()

# Text Cleaning

###Making statement text in lower case

In [ ]:
df['statement']=df['statement'].str.lower()
df['statement'].tail()

### Cleaning and removing Stop words of english

In [ ]:
import nltk
nltk.download('stopwords')

In [ ]:
from nltk.corpus import stopwords
", ".join(stopwords.words('english'))

In [8]:
stopwords_list = stopwords.words('english')

Cleaning and removing the above stop words list from the statement of news

In [ ]:
STOPWORDS = set(stopwords.words('english'))
def cleaning_stopwords(text):
    return " ".join([word for word in str(text).split() if word not in STOPWORDS])
df["statement"] = df["statement"].apply(lambda text: cleaning_stopwords(text))
df["statement"].head()

### Cleaning and removing punctuations

In [10]:
import string
english_punctuations = string.punctuation
punctuations_list = english_punctuations
def cleaning_punctuations(text):
    translator = str.maketrans('', '', punctuations_list)
    return text.translate(translator)

In [ ]:
df["statement"] = df["statement"].apply(lambda x: cleaning_punctuations(x))
df["statement"].tail()

### Cleaning and removing repeating characters

In [12]:
import re

In [13]:
def cleaning_repeating_char(text):
    return re.sub(r'(.)\1+', r'\1', text)

In [ ]:
df["statement"] = df["statement"].apply(lambda x: cleaning_repeating_char(x))
df["statement"].tail()

### Cleaning and removing email

In [15]:
def cleaning_email(data):
    return re.sub('@[^\s]+', ' ', data)

In [ ]:
df["statement"] = df["statement"].apply(lambda x: cleaning_email(x))
df["statement"].tail()

### Getting tokenization of news statement text

In [17]:
from nltk.tokenize import RegexpTokenizer

In [ ]:
tokenizer = RegexpTokenizer(r'\w+')
df["statement"] = df["statement"].apply(tokenizer.tokenize)
df["statement"].head()

### Applying Stemming

In [ ]:
st = nltk.PorterStemmer()
def stemming_on_text(data):
    text = [st.stem(word) for word in data]
    return data

df["statement"] = df["statement"].apply(lambda x: stemming_on_text(x))
df["statement"].head()

### Applying Lemmatizer

In [ ]:
import nltk
nltk.download('wordnet')

In [ ]:
lm = nltk.WordNetLemmatizer()
def lemmatizer_on_text(data):
    text = [lm.lemmatize(word) for word in data]
    return data

df["statement"] = df["statement"].apply(lambda x: lemmatizer_on_text(x))
df["statement"].head()

### Features extraction from the "Statement of the news"

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
word_vectorizer = TfidfVectorizer(
    sublinear_tf=True,
    strip_accents='unicode',
    analyzer='word',
    token_pattern=r'\w{1,}',
    ngram_range=(3, 3),
    max_features =5000)

Get_Vec= word_vectorizer.fit_transform(df['statement'].astype('str'))
Get_Vec= Get_Vec.toarray()

vocab1 = word_vectorizer.get_feature_names_out()
Features_vect=pd.DataFrame(np.round(Get_Vec, 1), columns=vocab1)
Features_vect.head()

Drop Statement Colunm

### Encodings of label, speaker, state info

In [ ]:
x = pd.Categorical(df['label'])
df['label']=x.codes


In [ ]:
import matplotlib
import seaborn as sns

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

### K fold cross validation and splitting the data five times

In [ ]:
from sklearn.model_selection import KFold

In [ ]:
kf = KFold(n_splits=5)
i=0
for train, test in kf.split(df):
    i=i+1
    print("KFold Split ",i )
    print("%s %s" % (train, test))
    print(' \n')

# Model Training

# Random Forest

In [ ]:
from cuml.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold
import numpy as np

X = Features_vect.values          # features
y = df['label'].values                    # your target column

kf = KFold(n_splits=5, shuffle=True, random_state=42)

fold = 0
accuracies = []

for train_index, test_index in kf.split(X):
    fold += 1
    print("KFold Split:", fold)

    # Split data
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Create and train model
    model = RandomForestClassifier()
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Accuracy
    acc = accuracy_score(y_test, y_pred)
    accuracies.append(acc)
    print("Accuracy on Fold", fold, ":", acc)
    print()

print("Mean Accuracy across folds:", np.mean(accuracies))
